# MVP03 — Modelagem de Dados

Nesta fase, será desenhado o modelo dimensional da camada Gold antes de qualquer movimentação de dados. Serão criadas tabelas vazias contendo o esquema, a descrição de cada coluna, o domínio de valores, a linhagem dos dados e as respectivas chaves. 

O notebook da fase MVP07-Gold será responsável pelo preenchimento dessas tabelas.

## Considerações de modelagem

**1. Esquema estrela no Gold.** As perguntas de negócio são analíticas
(agregações são pelo estado, categoria, mês e vendedor). O estrutura do modelo estrela  reduz a complexidade dos relacionamentos, facilita a exploração dos dados e acelera a geração de insights para a tomada de decisão.

**2. Duas tabelas de fato com diferentes grãos.**
- `fato_pedidos` — grão: **um pedido**. Responde às perguntas 1, 2, 3, 5 e 6,
  que tratam de prazo, atraso, avaliação, frete e pagamento — todos atributos
  do pedido como um todo.
- `fato_itens_pedido` — grão: **um item de pedido**. Responde às perguntas 4 e 7,
  que exigem cortar a receita por produto/categoria e por vendedor.

A separação dos grãos evita distorções analíticas. Sem essa abordagem, avaliações de pedidos seriam duplicadas em cada item, impactando indevidamente médias e indicadores. Com fatos distintas, as métricas permanecem consistentes e representativas da realidade do negócio.

**3. Chaves naturais em vez de chaves substitutas.** 
Optou-se pelo uso de chaves naturais, pois os identificadores da Olist já são únicos. A adoção de chaves substitutas agregaria complexidade ao modelo. Na dimensão tempo, a própria data foi utilizada como chave pra manter a simplicidade e a clareza da modelagem.


**4. `customer_id` × `customer_unique_id`.**

- `customer_id` é utilizado como chave por ser o identificador vinculado aos pedidos.
- `customer_unique_id` representa o cliente de forma única ao longo do tempo.
- Preserva a integridade do relacionamento com a tabela de fatos.
Permite contabilizar clientes corretamente, independentemente do número de compras.
- Viabiliza análises mais precisas de retenção, recorrência e comportamento do cliente.


**5. Exclusão da Tabela de Geolocalização do Modelo Analítico (Gold)** 
- ~1 milhão de registros com múltiplas coordenadas por prefixo de CEP.
- Requisitos analíticos atendidos com dados de UF já presentes nas dimensões de clientes e vendedores.
- Evitar aumento desnecessário da complexidade e do volume do modelo.
- Mantida na camada Bronze para rastreabilidade e governança dos dados.
- Foco em um modelo mais enxuto e alinhado às necessidades de negócio.

**6. Tradução de categorias incorporada à dimensão produto**
- Pra evitar snowflake.

## Diagrama do modelo
Esquema geral de como seria o modelo


    fato_pedidos            (1 linha = 1 pedido)
      ├── customer_id   →  dim_cliente
      └── data_compra   →  dim_tempo

    fato_itens_pedido       (1 linha = 1 item de pedido)
      ├── product_id    →  dim_produto
      ├── seller_id     →  dim_vendedor
      ├── customer_id   →  dim_cliente
      └── data_compra   →  dim_tempo

Diagrama elaborado no DrawSQL a partir do DDL das tabelas Gold. As linhas
representam relações um-para-muitos, com o lado "muitos" nas tabelas de fato.
`dim_cliente` e `dim_tempo` são dimensões conformadas, compartilhadas pelas
duas fatos.

![Diagrama estrella](./Images/modelo_estrella.PNG)


## Rastreabilidade: pergunta → tabelas

| Pergunta | Fato | Dimensões |
|---|---|---|
| 1. Variação × nota | fato_pedidos | — |
| 2. Prazo e frete por UF | fato_pedidos | dim_cliente |
| 3. Interestadual × intraestadual | fato_pedidos | — |
| 4. Receita e nota por categoria | fato_itens_pedido + fato_pedidos | dim_produto |
| 5. Evolução mensal | fato_pedidos | dim_tempo |
| 6. Parcelamento × ticket | fato_pedidos | — |
| 7. Concentração de vendedores | fato_itens_pedido | dim_vendedor |

In [0]:
%sql
-- Fatos primeiro (referenciam as dimensões), depois dimensões
DROP TABLE IF EXISTS mvp.gold.fato_itens_pedido;
DROP TABLE IF EXISTS mvp.gold.fato_pedidos;
DROP TABLE IF EXISTS mvp.gold.dim_tempo;
DROP TABLE IF EXISTS mvp.gold.dim_cliente;
DROP TABLE IF EXISTS mvp.gold.dim_vendedor;
DROP TABLE IF EXISTS mvp.gold.dim_produto;

In [0]:
%sql
CREATE TABLE mvp.gold.dim_tempo (
  data             DATE    NOT NULL COMMENT 'Data do calendário. Chave primária. Domínio: 2016-09-01 a 2018-10-31.',
  ano              INT     COMMENT 'Ano. Domínio: 2016 a 2018.',
  trimestre        INT     COMMENT 'Trimestre. Domínio: 1 a 4.',
  mes              INT     COMMENT 'Mês numérico. Domínio: 1 a 12.',
  nome_mes         STRING  COMMENT 'Nome do mês em português. Domínio: janeiro a dezembro.',
  ano_mes          STRING  COMMENT 'Ano-mês no formato AAAA-MM, para séries temporais.',
  dia_semana       INT     COMMENT 'Dia da semana. Domínio: 1 (domingo) a 7 (sábado).',
  nome_dia_semana  STRING  COMMENT 'Nome do dia da semana em português.',
  flag_fim_semana  BOOLEAN COMMENT 'Verdadeiro para sábado e domingo.',
  CONSTRAINT pk_dim_tempo PRIMARY KEY (data)
)
COMMENT 'Dimensão calendário. Grão: dia. ';

In [0]:
%sql
CREATE TABLE mvp.gold.dim_cliente (
  customer_id         STRING NOT NULL COMMENT 'Identificador do cliente no pedido; muda a cada compra. Chave primária.',
  customer_unique_id  STRING COMMENT 'Identificador estável da pessoa, permite contar clientes únicos.',
  cep_prefixo         STRING COMMENT 'Cinco primeiros dígitos do CEP, em texto para preservar zeros à esquerda.',
  cidade              STRING COMMENT 'Cidade padronizada.',
  uf                  STRING COMMENT 'Sigla do estado.',
  regiao              STRING COMMENT 'Região derivada da UF. Norte, Nordeste, Centro-Oeste, Sudeste, Sul.',
  CONSTRAINT pk_dim_cliente PRIMARY KEY (customer_id)
)
COMMENT 'Dimensão cliente. Grão: um customer_id.';

In [0]:
%sql
CREATE TABLE mvp.gold.dim_vendedor (
  seller_id    STRING NOT NULL COMMENT 'Identificador do vendedor. Chave primária.',
  cep_prefixo  STRING COMMENT 'Cinco primeiros dígitos do CEP do vendedor.',
  cidade       STRING COMMENT 'Cidade padronizada.',
  uf           STRING COMMENT 'Sigla do estado.',
  regiao       STRING COMMENT 'Região derivada da UF. Norte, Nordeste, Centro-Oeste, Sudeste, Sul.',
  CONSTRAINT pk_dim_vendedor PRIMARY KEY (seller_id)
)
COMMENT 'Dimensão vendedor. Grão: um vendedor.';

In [0]:
%sql
CREATE TABLE mvp.gold.dim_produto (
  product_id      STRING NOT NULL COMMENT 'Identificador do produto. Chave primária.',
  categoria       STRING COMMENT 'Categoria em português. Nulos na origem substituídos por "sem_categoria".',
  categoria_en    STRING COMMENT 'Categoria em inglês.',
  peso_g          INT    COMMENT 'Peso em gramas.',
  comprimento_cm  INT    COMMENT 'Comprimento em cm. Domínio esperado: > 0.',
  altura_cm       INT    COMMENT 'Altura em cm. Domínio esperado: > 0.',
  largura_cm      INT    COMMENT 'Largura em cm. Domínio esperado: > 0.',
  volume_cm3      INT    COMMENT 'Volume derivado: comprimento x altura x largura.',
  qtd_fotos       INT    COMMENT 'Quantidade de fotos no anúncio.',
  CONSTRAINT pk_dim_produto PRIMARY KEY (product_id)
)
COMMENT 'Dimensão produto. Grão: um produto.';

In [0]:
%sql
CREATE TABLE mvp.gold.fato_pedidos (
  order_id                  STRING NOT NULL COMMENT 'Identificador do pedido. Chave primária.',
  customer_id               STRING COMMENT 'FK para dim_cliente.',
  data_compra               DATE   COMMENT 'Data da compra. FK para dim_tempo.',
  status_pedido             STRING COMMENT 'Status do pedido. delivered, shipped, canceled, unavailable, invoiced, processing, created, approved.',
  data_hora_compra          TIMESTAMP COMMENT 'Momento da compra.',
  data_entrega              TIMESTAMP COMMENT 'Entrega ao cliente. Nulo se não entregue.',
  data_estimada             TIMESTAMP COMMENT 'Data de entrega prometida.',
  qtd_itens                 INT    COMMENT 'Número de itens do pedido.',
  qtd_vendedores            INT    COMMENT 'Número de vendedores distintos no pedido.',
  valor_produtos            DECIMAL(12,2) COMMENT 'Soma dos preços dos itens, em BRL.',
  valor_frete               DECIMAL(12,2) COMMENT 'Soma dos fretes, em BRL.',
  valor_total               DECIMAL(12,2) COMMENT 'valor_produtos + valor_frete, em BRL.',
  valor_pago                DECIMAL(12,2) COMMENT 'Total efetivamente pago, em BRL.',
  tipo_pagamento_principal  STRING COMMENT 'Meio de pagamento de maior valor no pedido. credit_card, boleto, voucher, debit_card.',
  qtd_parcelas              INT    COMMENT 'Maior número de parcelas entre os pagamentos do pedido.',
  dias_entrega              INT    COMMENT 'Dias entre compra e entrega. Nulo se não entregue.',
  dias_atraso               INT    COMMENT 'Dias entre entrega real e estimada.',
  flag_atraso               BOOLEAN COMMENT 'Verdadeiro se dias_atraso > 0.',
  flag_interestadual        BOOLEAN COMMENT 'Verdadeiro se ao menos um vendedor do pedido está em UF diferente da do cliente.',
  nota_avaliacao            INT    COMMENT 'Nota da avaliação. Domínio: 1 a 5. Nulo se sem avaliação.',
  CONSTRAINT pk_fato_pedidos PRIMARY KEY (order_id),
  CONSTRAINT fk_pedidos_cliente FOREIGN KEY (customer_id) REFERENCES mvp.gold.dim_cliente(customer_id),
  CONSTRAINT fk_pedidos_tempo   FOREIGN KEY (data_compra) REFERENCES mvp.gold.dim_tempo(data)
)
COMMENT 'Fato de pedidos. Grão: um pedido.';

In [0]:
%sql
CREATE TABLE mvp.gold.fato_itens_pedido (
  order_id            STRING NOT NULL COMMENT 'Identificador do pedido. Parte da chave primária.',
  order_item_id       INT    NOT NULL COMMENT 'Sequencial do item dentro do pedido.',
  product_id          STRING COMMENT 'FK para dim_produto.',
  seller_id           STRING COMMENT 'FK para dim_vendedor.',
  customer_id         STRING COMMENT 'FK para dim_cliente. Herdado do pedido.',
  data_compra         DATE   COMMENT 'FK para dim_tempo. Herdado do pedido.',
  preco               DECIMAL(10,2) COMMENT 'Preço do item em BRL.',
  valor_frete         DECIMAL(10,2) COMMENT 'Frete do item em BRL.',
  flag_interestadual  BOOLEAN COMMENT 'Verdadeiro se a UF do vendedor difere da UF do cliente.',
  CONSTRAINT pk_fato_itens PRIMARY KEY (order_id, order_item_id),
  CONSTRAINT fk_itens_produto  FOREIGN KEY (product_id)  REFERENCES mvp.gold.dim_produto(product_id),
  CONSTRAINT fk_itens_vendedor FOREIGN KEY (seller_id)   REFERENCES mvp.gold.dim_vendedor(seller_id),
  CONSTRAINT fk_itens_cliente  FOREIGN KEY (customer_id) REFERENCES mvp.gold.dim_cliente(customer_id),
  CONSTRAINT fk_itens_tempo    FOREIGN KEY (data_compra) REFERENCES mvp.gold.dim_tempo(data)
)
COMMENT 'Fato de itens de pedido. Grão: um item de pedido.';

In [0]:
%sql
SELECT table_name, column_name, data_type, comment
FROM mvp.information_schema.columns
WHERE table_schema = 'gold'
ORDER BY table_name, ordinal_position;